# 第14章 傅里叶变换（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 14 章跟随代码。**全量收录原书代码**（例 14.1~例 14.6 + 章内独立演示），缺失部分已按书补全，代码以网页笔记 docs/opencv/ch14.md 为准（含笔误与参数修正）。
> `cv2.imshow` 需在**本地 Python 环境**运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 `imread` 返回 None。
> 素材替换：原书 lena.bmp 用 picture2.png 顶替（例 14.1、14.2），boat.bmp 用 picture8.jpg 顶替（例 14.3~14.6）；例 14.6 低通掩模边长由原书的 30 调整为 50（中心 100×100 区域），与跟随代码一致。

# 第十四章 傅里叶变换

> 傅里叶变换把图像从空间域（spatial domain）搬到频率域（frequency domain）：颜色趋于一致的大块缓变区域是低频，灰度急剧过渡的边缘和噪声是高频；频域里低通滤波抹掉高频（图像变模糊、噪声减弱），高通滤波只留高频（边缘轮廓凸显），处理完再用逆傅里叶变换搬回空间域。频率、幅度（amplitude）、相位（phase）三个词是理解全章的钥匙。全章例题：例 14.1~14.3（Numpy 实现）、例 14.4~14.6（OpenCV 实现）。

## 14.2 Numpy 实现傅里叶变换

Numpy 模块提供了傅里叶变换功能，`numpy.fft.fft2()` 可以实现图像的傅里叶变换：输入**灰度图像**，返回复数数组（complex ndarray），零频率分量位于频谱左上角。本节用它实现变换与逆变换（例 14.1、14.2），并在频域内屏蔽低频、保留高频实现高通滤波（例 14.3）。

### 14.2.1 实现傅里叶变换

```text
返回值 = numpy.fft.fft2(原始图像)
返回值 = numpy.fft.fftshift(原始频谱)
```

两点注意：参数"原始图像"的类型是**灰度图像**；返回值是一个复数数组。零频率分量默认位于频谱左上角，用 `fftshift()` 把它移到频域图像的中心，便于观察。显示前要做缩放：`像素新值 = 20 × np.log(np.abs(频谱值))`——频谱值的动态范围极大（picture8 的直流分量约 8.6×10⁷），对数把"乘法尺度"压成"加法尺度"，大小悬殊的分量才能同时看得见。

### 例 14.1 · Numpy 傅里叶变换，观察频谱

左图是原始图像，右图是频谱图像：中心最亮的点就是零频率分量（直流分量 DC，等于图像全部像素值之和，对应图像的整体亮度），从中心向外频率越来越高，越亮的点表示该频率分量的幅度越大。

In [ ]:
# ===== 14.2.1 例 14.1：Numpy 傅里叶变换，观察频谱 =====
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("picture2.png", cv2.IMREAD_GRAYSCALE)   # 读入灰度图（原书 lena.bmp）

f = np.fft.fft2(img)                                     # 傅里叶变换：返回复数数组，零频率在左上角
fshift = np.fft.fftshift(f)                              # 零频率分量移到频谱中心
magnitude_spectrum = 20*np.log(np.abs(fshift))           # 取模 + 对数缩放，映射到可显示范围

plt.subplot(121), plt.imshow(img, cmap='gray')           # 左图：原始图像
plt.title("original"), plt.axis('off')
plt.subplot(122), plt.imshow(magnitude_spectrum, cmap='gray')   # 右图：频谱图像
plt.title("magnitude spectrum"), plt.axis('off')
plt.show()


### 14.2.2 实现逆傅里叶变换

逆傅里叶变换用 `numpy.fft.ifft2()` 实现，它是 `numpy.fft.fft2()` 的逆函数。注意：如果变换时用了 `fftshift()` 移动零频率分量，逆变换前要先用 `numpy.fft.ifftshift()` 把零频率分量移回原来的位置——fftshift 与 ifftshift 要**成对使用**。返回值仍旧是复数数组，用 `np.abs()` 取幅度调整到灰度空间：`iimg = np.abs(逆傅里叶变换结果)`。

### 例 14.2 · Numpy 傅里叶变换 + 逆变换

"fft2 → fftshift 移中 → ifftshift 移回 → ifft2"一圈回来的图像与原图肉眼看不出差别（实测最大误差约 2.8×10⁻¹³，纯属浮点精度）。

In [ ]:
# ===== 14.2.2 例 14.2：Numpy 傅里叶变换 + 逆变换 =====
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("picture2.png", cv2.IMREAD_GRAYSCALE)   # 原书 lena.bmp

f = np.fft.fft2(img)                  # 傅里叶变换
fshift = np.fft.fftshift(f)           # 零频率移到中心

ishift = np.fft.ifftshift(fshift)     # 逆变换前：零频率移回左上角
iimg = np.fft.ifft2(ishift)           # 逆傅里叶变换：返回空域复数数组
iimg = np.abs(iimg)                   # 取幅度，调整到灰度空间

plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title("original"), plt.axis("off")
plt.subplot(122), plt.imshow(iimg, cmap='gray')
plt.title("inverse FFT"), plt.axis("off")
plt.show()


### 14.2.3 高通滤波示例

低频信号对应图像内变化缓慢的灰度分量（大草原上颜色趋于一致的广袤草原），高频信号对应图像内变化越来越快的灰度分量（狮子的轮廓等边缘与噪声）。**高通滤波器**让高频信号通过、使低频信号衰减，会增强图像中尖锐的细节、降低对比度。做法：变换后低频信号位于频谱的中心位置，把频谱中心区域的低频分量值都替换为 0（处理为黑色），处理完再逆变换回空域：

```python
rows, cols = img.shape                        # 图像的行数、列数
crow, ccol = int(rows/2), int(cols/2)         # 频谱中心坐标
fshift[crow-30:crow+30, ccol-30:ccol+30] = 0  # 中心 60×60 区域的低频分量置零
```

> 注意：置零后若再对频谱取对数显示，`np.log(0)` 会产生 -inf 并触发 RuntimeWarning，要用 `20*np.log(np.abs(...) + 1)` 这类"加一防零"写法。

### 例 14.3 · Numpy 高通滤波（频谱中心置零）

把频谱中心 60×60 的低频分量（含直流分量）置零后逆变换：边缘信息得以保留，大面积平缓的区域（低频）被抹掉，还原图只剩轮廓感、整体偏暗。实测 picture8：中心 60×60 区域的频谱幅度总和占 12.2%，按平方能量算占 98.1%——自然图像的能量高度集中在低频。

In [ ]:
# ===== 14.2.3 例 14.3：Numpy 高通滤波（频谱中心置零） =====
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("picture8.jpg", cv2.IMREAD_GRAYSCALE)   # 原书 boat.bmp

f = np.fft.fft2(img)                             # 傅里叶变换
fshift = np.fft.fftshift(f)                      # 零频率移到中心

# --- 跟随代码增补：滤波前先看一眼频谱 ---
magnitude_spectrum = 20*np.log(np.abs(fshift))
plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title("original"), plt.axis('off')
plt.subplot(122), plt.imshow(magnitude_spectrum, cmap='gray')
plt.title("magnitude spectrum"), plt.axis('off')
plt.show()

rows, cols = img.shape                           # 行数、列数
crow, ccol = int(rows/2), int(cols/2)            # 频谱中心坐标
fshift[crow-30:crow+30, ccol-30:ccol+30] = 0     # 中心 60×60 低频置零 → 高通滤波

ishift = np.fft.ifftshift(fshift)                # 零频率移回原位
iimg = np.fft.ifft2(ishift)                      # 逆傅里叶变换
iimg = np.abs(iimg)                              # 取幅度

plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title("original"), plt.axis('off')
plt.subplot(122), plt.imshow(iimg, cmap='gray')
plt.title("inverse"), plt.axis('off')
plt.show()


## 14.3 用 OpenCV 实现傅里叶变换

OpenCV 提供了 `cv2.dft()` 和 `cv2.idft()` 来实现傅里叶变换和逆傅里叶变换。与 Numpy 版的差别主要在"装频谱的容器"：cv2.dft 的返回值是**双通道**数组——第 1 个通道是结果的实数部分，第 2 个通道是虚数部分（两者的数值结果与 np.fft.fft2 一致）。使用要点：

- 输入图像要先 `np.float32()` 转换成 np.float32 格式（uint8 直接断言失败）；
- 转换标识通常取 `cv2.DFT_COMPLEX_OUTPUT`，用来输出一个复数阵列；
- 幅度用 `cv2.magnitude(实部, 虚部)` 计算——平方和的平方根，两个参数必须同 size。

### 14.3.1 实现傅里叶变换

```text
返回结果 = cv2.dft(原始图像, 转换标识)
```

变换后零频率分量同样不在中心，要将其移至中心：

```python
dftShift = np.fft.fftshift(dft)    # 把频谱 dft 中的零频率分量移到中心
```

幅度对数缩放：幅度对数缩放：`result = 20 × np.log(cv2.magnitude(实部, 虚部))`。注意 20·ln 只是把动态范围压缩到几百的跨度，**并不保证落在 [0, 255] 内**（实测 picture8 的 result 范围 [51.5, 365.5]），显示正常是因为 plt.imshow 默认按数据 min–max 归一化。

**过程演示**（原书 14.3.1）：对图像做傅里叶变换、计算幅度并做规范化处理，打印各步骤的值（dft 原始值 → fftshift 移中 → 20·log 幅度），观察每一步的取值范围。

In [ ]:
# ===== 14.3.1 演示：观察 cv2.dft 各步骤的值 =====
import numpy as np
import cv2

img = cv2.imread('picture8.jpg', cv2.IMREAD_GRAYSCALE)           # 原书 lena.bmp
dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)     # 转 float32 后变换，输出双通道
print(dft)                                                       # 频谱原始值：实部+虚部
dftShift = np.fft.fftshift(dft)                                  # 零频率分量移到中心
print(dftShift)
result = 20*np.log(cv2.magnitude(dftShift[:, :, 0], dftShift[:, :, 1]))   # 幅度 → 对数缩放
print(result)


### 例 14.4 · OpenCV 傅里叶变换，展示频谱

左图是原始图像，右图是频谱图像——用 `np.fft.fftshift()` 把零频率分量移至中心后的幅度谱，形态与例 14.1 的 Numpy 版一致。

In [ ]:
# ===== 14.3.1 例 14.4：OpenCV 傅里叶变换，展示频谱 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture8.jpg', cv2.IMREAD_GRAYSCALE)           # 原书 lena.bmp

dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)     # 傅里叶变换：float32 输入、双通道输出
dftShift = np.fft.fftshift(dft)                                  # 零频率分量移到中心
result = 20*np.log(cv2.magnitude(dftShift[:, :, 0], dftShift[:, :, 1]))   # 幅度 → 对数缩放

plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title('Input Image'), plt.axis('off')
plt.subplot(122), plt.imshow(result, cmap='gray')
plt.title('Magnitude Spectrum'), plt.axis('off')
plt.show()


### 14.3.2 实现逆傅里叶变换

OpenCV 用 `cv2.idft()` 实现逆傅里叶变换，它是 `cv2.dft()` 的逆函数。如果用了 `np.fft.fftshift()` 移动零频率分量，逆变换前要用 `np.fft.ifftshift()` 把零频率分量恢复到原来位置；逆变换后得到的值仍旧是复数（双通道），需要用 `cv2.magnitude()` 计算其幅度。

> **注意（idft 默认不缩放）**：cv2.dft 与 cv2.idft 默认都不做归一化，idft 的结果是原图的 N 倍（N = 行×列；8×8 数组实测被放大 64 倍）。原书例 14.5 直接 plt.imshow 看起来正常，是 plt.imshow 自动 min–max 归一化的功劳；换成 cv2.imshow 或 imwrite 就是一片全白。要拿真值，写 `cv2.idft(ishift, flags=cv2.DFT_SCALE)`。

### 例 14.5 · OpenCV 傅里叶变换 + 逆变换

"dft → fftshift 移中 → ifftshift 移回 → idft → magnitude 取幅度"一圈回来，右图与原图肉眼一致。

In [ ]:
# ===== 14.3.2 例 14.5：OpenCV 傅里叶变换 + 逆变换 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture8.jpg', cv2.IMREAD_GRAYSCALE)           # 原书 lena.bmp

dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)     # 傅里叶变换
dftShift = np.fft.fftshift(dft)                                  # 零频率移到中心
ishift = np.fft.ifftshift(dftShift)                              # 零频率移回原位
iImg = cv2.idft(ishift)                                          # 逆傅里叶变换：返回复数（双通道）
iImg = cv2.magnitude(iImg[:, :, 0], iImg[:, :, 1])               # 计算幅度

plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title('original'), plt.axis('off')
plt.subplot(122), plt.imshow(iImg, cmap='gray')
plt.title('inverse'), plt.axis('off')
plt.show()


### 14.3.3 低通滤波示例

**低通滤波器**让低频信号通过、使高频信号衰减，图像低通滤波后会变模糊。做法：构造一个 0/1 掩模图像，用它**乘**频谱图像，把频谱中高频信号的值都替换为 0（黑）：

```python
rows, cols = img.shape                       # 行数、列数
crow, ccol = int(rows/2), int(cols/2)        # 中心坐标
mask = np.zeros((rows, cols, 2), np.uint8)   # 两个通道，与频域图像匹配
mask[crow-30:crow+30, ccol-30:ccol+30] = 1   # 中心 60×60 置 1，其余为 0
fShift = dftShift * mask                     # 掩模乘频谱
```

> **勘误**：原书"用掩模与频谱图像进行与运算"应为**乘法运算**（实际代码 `fShift = dftShift * mask`）——0/1 掩模下乘法与按位与的结果恰好一致，但按位与对浮点频谱并无定义。掩模必须是 (rows, cols, 2)，与 cv2.dft 的双通道频谱匹配。跟随代码把掩模边长由原书的 30 调整为 50（中心 100×100）。

### 例 14.6 · OpenCV 低通滤波（掩模法）

掩模中心 100×100 置 1（原书为 30，跟随代码改 50）：高频归零、低频保留，逆变换后图像的边缘信息被削弱、整体变模糊。实测 picture8：保留区域的频谱幅度总和只剩 20.7%，但按平方能量算仍占 98.6%——直流分量稳稳留在中心，整图均值几乎不变（213.28 vs 213.27，与高通"连亮度一起删"形成对照）。

In [ ]:
# ===== 14.3.3 例 14.6：OpenCV 低通滤波（掩模法） =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture8.jpg', cv2.IMREAD_GRAYSCALE)           # 原书 lena.bmp

dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)     # 傅里叶变换
dftShift = np.fft.fftshift(dft)                                  # 零频率移到中心

# --- 跟随代码增补：滤波前先看一眼频谱 ---
result = 20*np.log(cv2.magnitude(dftShift[:, :, 0], dftShift[:, :, 1]))
plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title('Input Image'), plt.axis('off')
plt.subplot(122), plt.imshow(result, cmap='gray')
plt.title('Magnitude Spectrum'), plt.axis('off')
plt.show()

rows, cols = img.shape
crow, ccol = int(rows/2), int(cols/2)
mask = np.zeros((rows, cols, 2), np.uint8)   # 两个通道，与频域图像匹配
mask[crow-50:crow+50, ccol-50:ccol+50] = 1   # 中心 100×100 置 1（原书为 30，跟随代码改 50）
fShift = dftShift * mask                     # 掩模乘频谱：中心低频保留，四周高频归零

# --- 跟随代码增补：滤波后的频谱（+1 防 log(0) 产生 -inf） ---
magnitude_spectrum1 = 20*np.log(cv2.magnitude(fShift[:, :, 0], fShift[:, :, 1]) + 1)
plt.imshow(magnitude_spectrum1, cmap='gray')
plt.title('Magnitude Spectrum after filtering'), plt.axis('off')
plt.show()

ishift = np.fft.ifftshift(fShift)            # 零频率移回原位
iImg = cv2.idft(ishift)                      # 逆傅里叶变换
iImg = cv2.magnitude(iImg[:, :, 0], iImg[:, :, 1])   # 取幅度

plt.subplot(121), plt.imshow(img, cmap='gray')
plt.title('original'), plt.axis('off')
plt.subplot(122), plt.imshow(iImg, cmap='gray')
plt.title('inverse'), plt.axis('off')
plt.show()
